# CT107-3-3-TXSA — Group Assignment Part A
## Member 2 — Q2: Word Stemming & Q5: Alternative Approach (SentencePiece)
**Dataset:** Data_1.txt

---
## Imports & Setup

In [ ]:
# Install required libraries (run once)
# !pip install nltk sentencepiece

import nltk
import re
import string
import sentencepiece as spm
import os

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer

# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

print('All imports and downloads complete.')

---
## Load Data_1.txt

In [ ]:
# Load the dataset
with open('Data_1.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print('Raw Text:')
print(raw_text)

---
## Pre-processing (Tokenization + Stop Word Removal)
Using the group's selected method (NLTK word_tokenize) as per Section 1.2.2 of the group report.

In [ ]:
# Tokenize using NLTK (group's chosen method — Section 1.2.2)
tokens = word_tokenize(raw_text.lower())

# Define noise set: NLTK stop words + punctuation
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)
noise = stop_words.union(punctuation)

# Filter tokens
filtered_tokens = [word for word in tokens if word not in noise]

print('Filtered Tokens (input for stemming) ~~~~~~~~~~~~~~~~~~~~~')
print(filtered_tokens)

---
# Q2 — Word Stemming
## Q2.1 — Importance of Stemming
*(Refer to report write-up — Section 2.1)*

---
## Q2.2 — Word Stemming Implementation

### 2.2.1 — Regular Expression Stemmer

In [ ]:
print('--- Q2.2.1: Regular Expression Stemmer ---')
print()

# RegexpStemmer strips suffixes matching the given pattern
# Rule: remove common English suffixes
regex_stemmer = RegexpStemmer('ing$|ed$|ion$|ions$|tion$|tions$|s$|ly$|er$|est$', min=4)

regex_stemmed = [regex_stemmer.stem(word) for word in filtered_tokens]

print('Stemming using Regular Expression Stemmer ~~~~~~~~~~~~~~~~~')
print(regex_stemmed)
print()

# Display word-to-stem mapping
print('Word → Stem Mapping:')
for original, stemmed in zip(filtered_tokens, regex_stemmed):
    if original != stemmed:
        print(f'  {original:30s} → {stemmed}')

### 2.2.2 — Porter Stemmer

In [ ]:
print('--- Q2.2.2: Porter Stemmer ---')
print()

# PorterStemmer: iterative suffix-stripping algorithm (moderate aggressiveness)
porter = PorterStemmer()

porter_stemmed = [porter.stem(word) for word in filtered_tokens]

print('Stemming using Porter Stemmer ~~~~~~~~~~~~~~~~~')
print(porter_stemmed)
print()

# Display word-to-stem mapping
print('Word → Stem Mapping:')
for original, stemmed in zip(filtered_tokens, porter_stemmed):
    if original != stemmed:
        print(f'  {original:30s} → {stemmed}')

### 2.2.3 — Lancaster Stemmer

In [ ]:
print('--- Q2.2.3: Lancaster Stemmer ---')
print()

# LancasterStemmer: iterative table-driven algorithm (most aggressive)
lancaster = LancasterStemmer()

lancaster_stemmed = [lancaster.stem(word) for word in filtered_tokens]

print('Stemming using Lancaster Stemmer ~~~~~~~~~~~~~~~~~')
print(lancaster_stemmed)
print()

# Display word-to-stem mapping
print('Word → Stem Mapping:')
for original, stemmed in zip(filtered_tokens, lancaster_stemmed):
    if original != stemmed:
        print(f'  {original:30s} → {stemmed}')

---
## Q2.3 — Comparative Analysis of Stemming Results

In [ ]:
print('--- Q2.3: Comparative Analysis ---')
print()
print(f'{"Original Token":<25} {"Regex Stem":<25} {"Porter Stem":<25} {"Lancaster Stem":<25}')
print('-' * 100)

for orig, reg, por, lan in zip(filtered_tokens, regex_stemmed, porter_stemmed, lancaster_stemmed):
    print(f'{orig:<25} {reg:<25} {por:<25} {lan:<25}')

---
# Q5 — Alternative Approach: SentencePiece Tokenizer

## 5.2.4 — SentencePiece Implementation

### Step 1 — Train SentencePiece Model on Data_1.txt

In [ ]:
print('--- Q5: SentencePiece Tokenizer ---')
print()

# Write raw text to a training file (SentencePiece requires a file input)
with open('sp_train.txt', 'w', encoding='utf-8') as f:
    f.write(raw_text)

# Train SentencePiece BPE model directly on Data_1.txt
# vocab_size set small (50) to suit the small corpus
spm.SentencePieceTrainer.train(
    input='sp_train.txt',
    model_prefix='sp_model',
    vocab_size=50,
    model_type='bpe',           # Byte Pair Encoding algorithm
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    character_coverage=1.0      # Full coverage of all characters in corpus
)

print('SentencePiece BPE model trained on Data_1.txt successfully.')

### Step 2 — Tokenize Data_1.txt Using Trained Model

In [ ]:
# Load the trained model
sp = spm.SentencePieceProcessor()
sp.load('sp_model.model')

# Tokenize raw text using the trained SentencePiece model
sp_tokens = sp.encode_as_pieces(raw_text)

print('Tokenisation using SentencePiece Tokenizer ~~~~~~~~~~~~~~~~~')
print(sp_tokens)
print()
print(f'Total tokens: {len(sp_tokens)}')

### Step 3 — Filter Stop Words & Punctuation

In [ ]:
print('--- Stop Words and Punctuation Filtering (SentencePiece) ---')
print()

# Normalise tokens: strip SentencePiece word-start marker (▁) for comparison
# The ▁ marker indicates the start of a new word in SentencePiece output
stop_words_sp = set(stopwords.words('english'))
punctuation_sp = set(string.punctuation)

def clean_sp_token(token):
    """Strip the SentencePiece word-boundary marker for stop word comparison."""
    return token.replace('\u2581', '').lower()  # ▁ is unicode U+2581

# Identify stop words found in the SentencePiece tokenized corpus
sp_stop_words_found = sorted(set(
    clean_sp_token(t) for t in sp_tokens
    if clean_sp_token(t) in stop_words_sp
))

print('List of Identified Stop Words in Data_1.txt:')
print(sp_stop_words_found)
print()

# Filter: remove stop words, punctuation, and whitespace tokens
sp_filtered = [
    t for t in sp_tokens
    if clean_sp_token(t) not in stop_words_sp
    and clean_sp_token(t) not in punctuation_sp
    and clean_sp_token(t).strip() != ''
]

print('Filtered Output (Noise Removed) ~~~~~~~~~~~~~~~~~')
print(sp_filtered)
print()
print(f'Summary: Original ({len(sp_tokens)} tokens) -> Filtered ({len(sp_filtered)} tokens)')

### Step 4 — Demonstrate Subword Segmentation

In [ ]:
# Highlight tokens that are subword pieces (do not start with ▁ and are not the first token)
# These demonstrate SentencePiece's BPE segmentation capability
print('Subword Segmentation Examples (tokens without ▁ are continuations):')
print()

subword_pieces = [t for t in sp_tokens if not t.startswith('\u2581') and t.strip() not in string.punctuation]

if subword_pieces:
    print('Continuation subword pieces found:')
    for piece in set(subword_pieces):
        print(f'  {piece}')
else:
    print('No subword continuations found — vocabulary large enough to cover all words as units.')

print()
print('Full vocabulary learned from Data_1.txt:')
vocab = [sp.id_to_piece(i) for i in range(sp.get_piece_size())]
print(vocab)